In [1]:
# ============================================
# Install TabM OFFLINE (FOR KAGGLE SUBMISSION!)
# ============================================
# Cài từ wheels dataset (KHÔNG CẦN INTERNET khi submit!)

# 📦 Dataset: tabm-kaggle-wheels (đã upload folder tabm_kaggle_wheels)
# ⚠️ CRITICAL: Chỉ cài tabm + rtdl_num_embeddings (Kaggle đã có PyTorch)
!pip install /kaggle/input/tabm-kaggle-wheels/tabm-0.0.3-py3-none-any.whl /kaggle/input/tabm-kaggle-wheels/rtdl_num_embeddings-0.0.12-py3-none-any.whl --no-deps -q

# ✅ Có thể SUBMIT với Internet = OFF!
# ✅ Không conflict vì không cài lại PyTorch

# NOTE: Nếu chưa upload dataset:
# 1. Upload folder 'tabm_kaggle_wheels' lên https://www.kaggle.com/datasets
# 2. Đặt tên: tabm-kaggle-wheels
# 3. Add dataset vào notebook inputs

In [2]:
import os
import math
import itertools
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
import polars as pl

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder, QuantileTransformer
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import tabm
import rtdl_num_embeddings

import warnings
warnings.filterwarnings("ignore")

# Set seeds for reproducibility
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## 1. Configuration & Feature Definitions

In [3]:
import os

# ============================================
# Auto-detect paths (KaggleHub download location)
# ============================================
# Use the path where kagglehub.competition_download extracts the data
root = Path(r"/kaggle/input/isic-2024-challenge")
print(f"Using data path: {root}")

train_path = root / 'train-metadata.csv'
test_path = root / 'test-metadata.csv'
subm_path = root / 'sample_submission.csv'

id_col = 'isic_id'
target_col = 'target'
group_col = 'patient_id'

err = 1e-5
seed = 42

num_cols = [
    'age_approx', 'clin_size_long_diam_mm',
    'tbp_lv_A', 'tbp_lv_Aext', 'tbp_lv_B', 'tbp_lv_Bext',
    'tbp_lv_C', 'tbp_lv_Cext', 'tbp_lv_H', 'tbp_lv_Hext',
    'tbp_lv_L', 'tbp_lv_Lext', 'tbp_lv_areaMM2', 'tbp_lv_area_perim_ratio',
    'tbp_lv_color_std_mean', 'tbp_lv_deltaA', 'tbp_lv_deltaB', 'tbp_lv_deltaL',
    'tbp_lv_deltaLB', 'tbp_lv_deltaLBnorm', 'tbp_lv_eccentricity',
    'tbp_lv_minorAxisMM', 'tbp_lv_nevi_confidence', 'tbp_lv_norm_border',
    'tbp_lv_norm_color', 'tbp_lv_perimeterMM', 'tbp_lv_radial_color_std_max',
    'tbp_lv_stdL', 'tbp_lv_stdLExt', 'tbp_lv_symm_2axis', 'tbp_lv_symm_2axis_angle',
    'tbp_lv_x', 'tbp_lv_y', 'tbp_lv_z',
]

new_num_cols = [
    'lesion_size_ratio', 'lesion_shape_index', 'hue_contrast', 'luminance_contrast',
    'lesion_color_difference', 'border_complexity', 'color_uniformity',
    'position_distance_3d', 'perimeter_to_area_ratio', 'area_to_perimeter_ratio',
    'lesion_visibility_score', 'symmetry_border_consistency', 'consistency_symmetry_border',
    'color_consistency', 'consistency_color', 'size_age_interaction',
    'hue_color_std_interaction', 'lesion_severity_index', 'shape_complexity_index',
    'color_contrast_index', 'log_lesion_area', 'normalized_lesion_size',
    'mean_hue_difference', 'std_dev_contrast', 'color_shape_composite_index',
    'lesion_orientation_3d', 'overall_color_difference',
    'symmetry_perimeter_interaction', 'comprehensive_lesion_index',
    'color_variance_ratio', 'border_color_interaction',
    'border_color_interaction_2', 'size_color_contrast_ratio',
    'age_normalized_nevi_confidence', 'age_normalized_nevi_confidence_2',
    'color_asymmetry_index', 'volume_approximation_3d', 'color_range',
    'shape_color_consistency', 'border_length_ratio', 'age_size_symmetry_index',
    'index_age_size_symmetry',
]

# Categorical columns (same as original)
cat_cols = ['sex', 'anatom_site_general', 'tbp_tile_type', 'tbp_lv_location', 'tbp_lv_location_simple']

# Patient-normalized columns
norm_cols = [f'{col}_patient_norm' for col in num_cols + new_num_cols]

# All continuous features for TabM
all_num_cols = num_cols + new_num_cols + norm_cols + ['count_per_patient']

Using data path: /kaggle/input/isic-2024-challenge


## 2. Data Loading & Feature Engineering

In [4]:
def read_data(path):
    return (
        pl.read_csv(path)
        .with_columns(
            pl.col('age_approx').cast(pl.String).replace('NA', np.nan).cast(pl.Float64),
        )
        .with_columns(
            pl.col(pl.Float64).fill_nan(pl.col(pl.Float64).median()),
        )
        # Feature Engineering
        .with_columns(
            lesion_size_ratio              = pl.col('tbp_lv_minorAxisMM') / pl.col('clin_size_long_diam_mm'),
            lesion_shape_index             = pl.col('tbp_lv_areaMM2') / (pl.col('tbp_lv_perimeterMM') ** 2),
            hue_contrast                   = (pl.col('tbp_lv_H') - pl.col('tbp_lv_Hext')).abs(),
            luminance_contrast             = (pl.col('tbp_lv_L') - pl.col('tbp_lv_Lext')).abs(),
            lesion_color_difference        = (pl.col('tbp_lv_deltaA') ** 2 + pl.col('tbp_lv_deltaB') ** 2 + pl.col('tbp_lv_deltaL') ** 2).sqrt(),
            border_complexity              = pl.col('tbp_lv_norm_border') + pl.col('tbp_lv_symm_2axis'),
            color_uniformity               = pl.col('tbp_lv_color_std_mean') / (pl.col('tbp_lv_radial_color_std_max') + err),
        )
        .with_columns(
            position_distance_3d           = (pl.col('tbp_lv_x') ** 2 + pl.col('tbp_lv_y') ** 2 + pl.col('tbp_lv_z') ** 2).sqrt(),
            perimeter_to_area_ratio        = pl.col('tbp_lv_perimeterMM') / pl.col('tbp_lv_areaMM2'),
            area_to_perimeter_ratio        = pl.col('tbp_lv_areaMM2') / pl.col('tbp_lv_perimeterMM'),
            lesion_visibility_score        = pl.col('tbp_lv_deltaLBnorm') + pl.col('tbp_lv_norm_color'),
            symmetry_border_consistency    = pl.col('tbp_lv_symm_2axis') * pl.col('tbp_lv_norm_border'),
            consistency_symmetry_border    = pl.col('tbp_lv_symm_2axis') * pl.col('tbp_lv_norm_border') / (pl.col('tbp_lv_symm_2axis') + pl.col('tbp_lv_norm_border')),
        )
        .with_columns(
            color_consistency              = pl.col('tbp_lv_stdL') / pl.col('tbp_lv_Lext'),
            consistency_color              = pl.col('tbp_lv_stdL') * pl.col('tbp_lv_Lext') / (pl.col('tbp_lv_stdL') + pl.col('tbp_lv_Lext')),
            size_age_interaction           = pl.col('clin_size_long_diam_mm') * pl.col('age_approx'),
            hue_color_std_interaction      = pl.col('tbp_lv_H') * pl.col('tbp_lv_color_std_mean'),
            lesion_severity_index          = (pl.col('tbp_lv_norm_border') + pl.col('tbp_lv_norm_color') + pl.col('tbp_lv_eccentricity')) / 3,
            shape_complexity_index         = pl.col('border_complexity') + pl.col('lesion_shape_index'),
            color_contrast_index           = pl.col('tbp_lv_deltaA') + pl.col('tbp_lv_deltaB') + pl.col('tbp_lv_deltaL') + pl.col('tbp_lv_deltaLBnorm'),
        )
        .with_columns(
            log_lesion_area                = (pl.col('tbp_lv_areaMM2') + 1).log(),
            normalized_lesion_size         = pl.col('clin_size_long_diam_mm') / pl.col('age_approx'),
            mean_hue_difference            = (pl.col('tbp_lv_H') + pl.col('tbp_lv_Hext')) / 2,
            std_dev_contrast               = ((pl.col('tbp_lv_deltaA') ** 2 + pl.col('tbp_lv_deltaB') ** 2 + pl.col('tbp_lv_deltaL') ** 2) / 3).sqrt(),
            color_shape_composite_index    = (pl.col('tbp_lv_color_std_mean') + pl.col('tbp_lv_area_perim_ratio') + pl.col('tbp_lv_symm_2axis')) / 3,
            lesion_orientation_3d          = pl.arctan2(pl.col('tbp_lv_y'), pl.col('tbp_lv_x')),
            overall_color_difference       = (pl.col('tbp_lv_deltaA') + pl.col('tbp_lv_deltaB') + pl.col('tbp_lv_deltaL')) / 3,
        )
        .with_columns(
            symmetry_perimeter_interaction = pl.col('tbp_lv_symm_2axis') * pl.col('tbp_lv_perimeterMM'),
            comprehensive_lesion_index     = (pl.col('tbp_lv_area_perim_ratio') + pl.col('tbp_lv_eccentricity') + pl.col('tbp_lv_norm_color') + pl.col('tbp_lv_symm_2axis')) / 4,
            color_variance_ratio           = pl.col('tbp_lv_color_std_mean') / pl.col('tbp_lv_stdLExt'),
            border_color_interaction       = pl.col('tbp_lv_norm_border') * pl.col('tbp_lv_norm_color'),
            border_color_interaction_2     = pl.col('tbp_lv_norm_border') * pl.col('tbp_lv_norm_color') / (pl.col('tbp_lv_norm_border') + pl.col('tbp_lv_norm_color')),
            size_color_contrast_ratio      = pl.col('clin_size_long_diam_mm') / pl.col('tbp_lv_deltaLBnorm'),
            age_normalized_nevi_confidence = pl.col('tbp_lv_nevi_confidence') / pl.col('age_approx'),
            age_normalized_nevi_confidence_2 = (pl.col('clin_size_long_diam_mm')**2 + pl.col('age_approx')**2).sqrt(),
            color_asymmetry_index          = pl.col('tbp_lv_radial_color_std_max') * pl.col('tbp_lv_symm_2axis'),
        )
        .with_columns(
            volume_approximation_3d        = pl.col('tbp_lv_areaMM2') * (pl.col('tbp_lv_x')**2 + pl.col('tbp_lv_y')**2 + pl.col('tbp_lv_z')**2).sqrt(),
            color_range                    = (pl.col('tbp_lv_L') - pl.col('tbp_lv_Lext')).abs() + (pl.col('tbp_lv_A') - pl.col('tbp_lv_Aext')).abs() + (pl.col('tbp_lv_B') - pl.col('tbp_lv_Bext')).abs(),
            shape_color_consistency        = pl.col('tbp_lv_eccentricity') * pl.col('tbp_lv_color_std_mean'),
            border_length_ratio            = pl.col('tbp_lv_perimeterMM') / (2 * np.pi * (pl.col('tbp_lv_areaMM2') / np.pi).sqrt()),
            age_size_symmetry_index        = pl.col('age_approx') * pl.col('clin_size_long_diam_mm') * pl.col('tbp_lv_symm_2axis'),
            index_age_size_symmetry        = pl.col('age_approx') * pl.col('tbp_lv_areaMM2') * pl.col('tbp_lv_symm_2axis'),
        )
        # Patient-level normalization - handle single patient cases
        .with_columns(
            pl.when(pl.col(col).count().over('patient_id') > 1)
            .then((pl.col(col) - pl.col(col).mean().over('patient_id')) / 
                  (pl.col(col).std().over('patient_id') + err))
            .otherwise(0.0)
            .alias(f'{col}_patient_norm')
            for col in (num_cols + new_num_cols)
        )
        # Count features
        .with_columns(
            pl.col(col).count().over('patient_id', col).alias(f'{col}_count') for col in cat_cols
        )
        .with_columns(
            count_per_patient = pl.col('isic_id').count().over('patient_id'),
        )
        .to_pandas()
    )

print("Loading data...")
df_train = read_data(train_path)
df_test = read_data(test_path)
df_subm = pd.read_csv(subm_path, index_col=id_col)

print(f"Train shape: {df_train.shape}")
print(f"Test shape: {df_test.shape}")
print(f"Positive samples: {df_train[target_col].sum()} ({df_train[target_col].mean()*100:.2f}%)")

Loading data...
Train shape: (401059, 179)
Test shape: (3, 168)
Positive samples: 393 (0.10%)


## 3. Preprocessing for TabM

In [5]:
# Build feature columns list (similar to original notebook)
cat_count_cols = [f'{col}_count' for col in cat_cols]
num_cat_interaction_cols = [f'{num_col}_{cat_col}' for num_col, cat_col in itertools.product(num_cols, cat_cols)]

# Note: For TabM, we'll use simplified feature set to avoid too many columns
# Using: numerical + engineered + normalized + cat_counts + count_per_patient
tabm_feature_cols = all_num_cols + cat_count_cols + ['count_per_patient']

print(f"Total TabM features: {len(tabm_feature_cols)}")
print(f"  - Numerical: {len(num_cols)}")
print(f"  - Engineered: {len(new_num_cols)}")
print(f"  - Patient normalized: {len(norm_cols)}")
print(f"  - Categorical counts: {len(cat_count_cols)}")
print(f"  - Count per patient: 1")

Total TabM features: 159
  - Numerical: 34
  - Engineered: 42
  - Patient normalized: 76
  - Categorical counts: 5
  - Count per patient: 1


In [6]:
# Handle missing/infinite values in all numerical features
def clean_numerical_features(df, cols):
    df = df.copy()
    for col in cols:
        if col in df.columns:
            df[col] = df[col].replace([np.inf, -np.inf], np.nan)
            df[col] = df[col].fillna(df[col].median())
    return df

# Fill missing count features with 0
for col in cat_count_cols:
    if col in df_train.columns:
        df_train[col] = df_train[col].fillna(0)
    if col in df_test.columns:
        df_test[col] = df_test[col].fillna(0)

df_train = clean_numerical_features(df_train, tabm_feature_cols)
df_test = clean_numerical_features(df_test, tabm_feature_cols)

# Extract numerical features (will be transformed)
X_num_train = df_train[tabm_feature_cols].values.astype(np.float32)
X_num_test = df_test[tabm_feature_cols].values.astype(np.float32)

# Add small noise to improve QuantileTransformer output
noise = np.random.default_rng(0).normal(0.0, 1e-5, X_num_train.shape).astype(np.float32)

quantile_transformer = QuantileTransformer(
    n_quantiles=max(min(len(df_train) // 30, 1000), 10),
    output_distribution='normal',
    subsample=10**9,
)
quantile_transformer.fit(X_num_train + noise)

X_num_train = quantile_transformer.transform(X_num_train).astype(np.float32)
X_num_test = quantile_transformer.transform(X_num_test).astype(np.float32)

# ============================================
# IMPROVED: USE REAL CATEGORICAL FEATURES
# ============================================
# Encode categorical features for TabM
cat_encoders = {}
X_cat_train_list = []
X_cat_test_list = []

for cat_col in cat_cols:
    le = LabelEncoder()
    
    # Fit on train + test combined to avoid unseen categories
    combined = pd.concat([
        df_train[cat_col].fillna('MISSING'),
        df_test[cat_col].fillna('MISSING')
    ])
    le.fit(combined)
    
    # Transform
    train_encoded = le.transform(df_train[cat_col].fillna('MISSING'))
    test_encoded = le.transform(df_test[cat_col].fillna('MISSING'))
    
    X_cat_train_list.append(train_encoded)
    X_cat_test_list.append(test_encoded)
    cat_encoders[cat_col] = le

# Stack all categorical features
X_cat_train = np.column_stack(X_cat_train_list).astype(np.int64)
X_cat_test = np.column_stack(X_cat_test_list).astype(np.int64)

# Get cardinalities for each categorical feature
cat_cardinalities = [len(cat_encoders[col].classes_) for col in cat_cols]

# Labels
y_train = df_train[target_col].values.astype(np.float32)

print(f"\n{'='*60}")
print("FEATURE SUMMARY")
print(f"{'='*60}")
print(f"X_num_train shape: {X_num_train.shape}")
print(f"X_cat_train shape: {X_cat_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"\nCategorical Features:")
for col, card in zip(cat_cols, cat_cardinalities):
    print(f"  {col}: {card} unique values")
print(f"\nMissing values in X_num_train: {np.isnan(X_num_train).sum()}")
print(f"Missing values in X_num_test: {np.isnan(X_num_test).sum()}")


FEATURE SUMMARY
X_num_train shape: (401059, 159)
X_cat_train shape: (401059, 5)
y_train shape: (401059,)

Categorical Features:
  sex: 3 unique values
  anatom_site_general: 6 unique values
  tbp_tile_type: 2 unique values
  tbp_lv_location: 21 unique values
  tbp_lv_location_simple: 8 unique values

Missing values in X_num_train: 0
Missing values in X_num_test: 0


## 4. Dataset & DataLoader

In [7]:
class ISICDataset(Dataset):
    def __init__(self, X_num, X_cat, y=None):
        self.X_num = torch.tensor(X_num, dtype=torch.float32)
        self.X_cat = torch.tensor(X_cat, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32) if y is not None else None

    def __len__(self):
        return len(self.X_num)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X_num[idx], self.X_cat[idx], self.y[idx]
        return self.X_num[idx], self.X_cat[idx]

## 5. Evaluation Metric (pAUC)

In [8]:
def comp_score(y_true, y_pred, min_tpr=0.80):
    """Calculate partial AUC score as used in ISIC 2024 competition"""
    v_gt = np.abs(np.asarray(y_true) - 1)
    v_pred = np.array([1.0 - x for x in y_pred])
    max_fpr = abs(1 - min_tpr)
    partial_auc_scaled = roc_auc_score(v_gt, v_pred, max_fpr=max_fpr)
    partial_auc = 0.5 * max_fpr**2 + (max_fpr - 0.5 * max_fpr**2) / (1.0 - 0.5) * (partial_auc_scaled - 0.5)
    return partial_auc

## 6. TabM Model Setup

In [9]:
# ============================================
# OPTIMIZED TabM Configuration
# ============================================
CONFIG = {
    # Model architecture
    'n_num_features': len(tabm_feature_cols),
    'cat_cardinalities': None,  # Will be set dynamically
    'd_out': 1,  # Binary classification
    
    # Numerical embeddings
    'n_bins': 96,  # ↑ Tăng từ 48 → 96 cho better granularity
    'd_embedding': 24,  # ↑ Tăng từ 16 → 24 cho richer representations
    
    # Categorical embeddings  
    'd_cat_embedding': 16,  # Embedding dimension cho categorical features
    
    # Training hyperparameters
    'batch_size': 512,  # ↑ Tăng từ 256 → 512 (faster, more stable)
    'lr': 3e-4,  # ↓ Giảm learning rate để train ổn định hơn
    'weight_decay': 5e-5,  # ↓ Giảm regularization
    'epochs': 150,  # ↑ Tăng max epochs
    'patience': 20,  # ↑ Tăng patience
    'gradient_clipping_norm': 2.0,  # ↑ Tăng gradient clipping
    
    # Learning rate schedule
    'warmup_epochs': 10,  # NEW: Warmup scheduler
    'min_lr': 1e-6,  # NEW: Minimum learning rate
    
    # Class imbalance
    'pos_weight': 1.648,  # From Optuna tuning
    
    # Mixup augmentation (TabM-specific)
    'mixup_alpha': 0.2,  # NEW: Mixup for regularization
    
    # Label smoothing
    'label_smoothing': 0.01,  # NEW: Prevent overconfidence
}

print("="*60)
print("OPTIMIZED TabM Configuration")
print("="*60)
for k, v in CONFIG.items():
    print(f"  {k}: {v}")
print("="*60)

OPTIMIZED TabM Configuration
  n_num_features: 159
  cat_cardinalities: None
  d_out: 1
  n_bins: 96
  d_embedding: 24
  d_cat_embedding: 16
  batch_size: 512
  lr: 0.0003
  weight_decay: 5e-05
  epochs: 150
  patience: 20
  gradient_clipping_norm: 2.0
  warmup_epochs: 10
  min_lr: 1e-06
  pos_weight: 1.648
  mixup_alpha: 0.2
  label_smoothing: 0.01


In [10]:
def create_tabm_model(X_num_train_tensor, cat_cardinalities):
    """Create OPTIMIZED TabM model with better embeddings"""

    # Compute bins for PiecewiseLinearEmbeddings with MORE bins
    bins = rtdl_num_embeddings.compute_bins(
        X_num_train_tensor,
        n_bins=CONFIG['n_bins']  # 96 instead of 48
    )

    # Create numerical embeddings with HIGHER dimension
    num_embeddings = rtdl_num_embeddings.PiecewiseLinearEmbeddings(
        bins,
        d_embedding=CONFIG['d_embedding'],  # 24 instead of 16
        activation=False,
        version='B',
    )

    # Create TabM model with categorical embeddings
    model = tabm.TabM.make(
        n_num_features=CONFIG['n_num_features'],
        cat_cardinalities=cat_cardinalities,
        d_out=CONFIG['d_out'],
        num_embeddings=num_embeddings,
        
        # TabM backbone parameters (use correct parameter names)
        # These are passed to make_tabm_backbone internally
        k=32,  # Number of ensemble members (default in TabM)
    )

    return model

# Update CONFIG with cat_cardinalities after encoding
CONFIG['cat_cardinalities'] = cat_cardinalities

# Test model creation
print("="*60)
print("Creating test model...")
print("="*60)
test_model = create_tabm_model(
    torch.tensor(X_num_train[:1000]), 
    cat_cardinalities
)
print(f"\n✅ TabM Model created successfully!")
print(f"   Number of ensemble members (K): {test_model.k}")
print(f"   Total parameters: {sum(p.numel() for p in test_model.parameters()):,}")
print(f"   Categorical embeddings: {len(cat_cardinalities)} features")
print(f"   Numerical embeddings: {CONFIG['n_num_features']} features")
del test_model

Creating test model...

✅ TabM Model created successfully!
   Number of ensemble members (K): 32
   Total parameters: 2,832,112
   Categorical embeddings: 5 features
   Numerical embeddings: 159 features


## 7. Training Functions

In [11]:
def mixup_data(X_num, X_cat, y, alpha=0.2):
    """Apply mixup augmentation for tabular data"""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = X_num.size(0)
    index = torch.randperm(batch_size).to(X_num.device)

    # Mix numerical features
    mixed_X_num = lam * X_num + (1 - lam) * X_num[index, :]
    
    # For categorical, randomly choose between original and shuffled
    mask = torch.rand(batch_size, X_cat.size(1)) < lam
    mixed_X_cat = torch.where(mask.to(X_cat.device), X_cat, X_cat[index, :])
    
    # Mix labels
    mixed_y = lam * y + (1 - lam) * y[index]
    
    return mixed_X_num, mixed_X_cat, mixed_y, lam


def train_epoch(model, train_loader, optimizer, criterion, device, 
                gradient_clipping_norm=None, mixup_alpha=0.0, label_smoothing=0.0):
    model.train()
    total_loss = 0

    for X_num, X_cat, y in train_loader:
        X_num, X_cat, y = X_num.to(device), X_cat.to(device), y.to(device)

        # Apply mixup augmentation
        if mixup_alpha > 0:
            X_num, X_cat, y, lam = mixup_data(X_num, X_cat, y, mixup_alpha)

        # Apply label smoothing
        if label_smoothing > 0:
            y = y * (1 - label_smoothing) + 0.5 * label_smoothing

        optimizer.zero_grad()

        # Forward pass: TabM outputs (batch, K, 1)
        y_pred = model(X_num, X_cat).squeeze(-1)  # (batch, K)

        # Train K predictions independently
        y_pred_flat = y_pred.flatten()
        y_flat = y.repeat_interleave(model.k)

        loss = criterion(y_pred_flat, y_flat)
        loss.backward()

        if gradient_clipping_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clipping_norm)

        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(train_loader)


@torch.no_grad()
def evaluate(model, data_loader, device):
    model.eval()
    all_preds = []
    all_targets = []

    for batch in data_loader:
        if len(batch) == 3:
            X_num, X_cat, y = batch
            all_targets.extend(y.numpy())
        else:
            X_num, X_cat = batch

        X_num, X_cat = X_num.to(device), X_cat.to(device)

        # Forward pass
        y_pred = model(X_num, X_cat).squeeze(-1)  # (batch, K)

        # Average K predictions (use probabilities for classification)
        probs = torch.sigmoid(y_pred)  # (batch, K)
        final_pred = probs.mean(dim=1)  # (batch,)

        all_preds.extend(final_pred.cpu().numpy())

    return np.array(all_preds), np.array(all_targets) if all_targets else None

## 8. Cross-Validation Training

In [12]:
N_SPLITS = 5
cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)

# Store results
fold_scores = []
fold_models = []
oof_predictions = np.zeros(len(df_train))

print("="*60)
print("Starting TabM Cross-Validation Training")
print("="*60)

Starting TabM Cross-Validation Training


In [13]:
for fold, (train_idx, val_idx) in enumerate(cv.split(df_train, df_train[target_col], groups=df_train[group_col])):
    print(f"\n{'='*60}")
    print(f"FOLD {fold + 1}/{N_SPLITS}")
    print(f"{'='*60}")
    print(f"Train size: {len(train_idx)}, Val size: {len(val_idx)}")

    # Split data
    X_num_tr, X_num_val = X_num_train[train_idx], X_num_train[val_idx]
    X_cat_tr, X_cat_val = X_cat_train[train_idx], X_cat_train[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]

    # Create datasets
    train_dataset = ISICDataset(X_num_tr, X_cat_tr, y_tr)
    val_dataset = ISICDataset(X_num_val, X_cat_val, y_val)

    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], 
                             shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], 
                           shuffle=False, num_workers=0, pin_memory=True)

    # Create model
    model = create_tabm_model(
        torch.tensor(X_num_tr), 
        CONFIG['cat_cardinalities']
    ).to(device)

    # Optimizer with better settings
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CONFIG['lr'],
        weight_decay=CONFIG['weight_decay'],
        betas=(0.9, 0.999),
        eps=1e-8
    )

    # Class weight for imbalance
    pos_weight = torch.tensor([CONFIG['pos_weight']]).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    # IMPROVED: Warmup + CosineAnnealing Scheduler
    warmup_epochs = CONFIG['warmup_epochs']
    total_epochs = CONFIG['epochs']
    
    # Warmup scheduler
    warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer, 
        start_factor=0.1, 
        end_factor=1.0, 
        total_iters=warmup_epochs
    )
    
    # Main scheduler (after warmup)
    cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, 
        T_max=total_epochs - warmup_epochs,
        eta_min=CONFIG['min_lr']
    )
    
    # Combine schedulers
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup_scheduler, cosine_scheduler],
        milestones=[warmup_epochs]
    )

    # Training loop
    best_score = -np.inf
    best_model_state = None
    patience_counter = 0

    for epoch in range(total_epochs):
        # Train with augmentations
        train_loss = train_epoch(
            model, train_loader, optimizer, criterion, device,
            gradient_clipping_norm=CONFIG['gradient_clipping_norm'],
            mixup_alpha=CONFIG['mixup_alpha'],
            label_smoothing=CONFIG['label_smoothing']
        )

        # Evaluate
        val_preds, val_targets = evaluate(model, val_loader, device)
        val_score = comp_score(val_targets, val_preds)

        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']

        # Check for improvement
        if val_score > best_score:
            best_score = val_score
            best_model_state = deepcopy(model.state_dict())
            patience_counter = 0
            marker = "⭐"
        else:
            patience_counter += 1
            marker = "  "

        if (epoch + 1) % 5 == 0 or marker == "⭐":
            print(f"{marker} Epoch {epoch+1:3d}: Loss={train_loss:.4f}, pAUC={val_score:.5f} "
                  f"(best={best_score:.5f}), LR={current_lr:.2e}")

        # Early stopping
        if patience_counter >= CONFIG['patience']:
            print(f"⏹️  Early stopping at epoch {epoch+1}")
            break

    # Load best model and get OOF predictions
    model.load_state_dict(best_model_state)
    val_preds, _ = evaluate(model, val_loader, device)
    oof_predictions[val_idx] = val_preds

    fold_scores.append(best_score)
    fold_models.append(model)

    print(f"\n✅ Fold {fold+1} Best pAUC: {best_score:.5f}")

print("\n" + "="*60)
print("CROSS-VALIDATION RESULTS")
print("="*60)
print(f"Fold scores: {[f'{s:.5f}' for s in fold_scores]}")
print(f"Mean pAUC: {np.mean(fold_scores):.5f} (+/- {np.std(fold_scores):.5f})")

# Overall OOF score
oof_score = comp_score(y_train, oof_predictions)
print(f"Overall OOF pAUC: {oof_score:.5f}")


FOLD 1/5
Train size: 329895, Val size: 71164
⭐ Epoch   1: Loss=0.1110, pAUC=0.07774 (best=0.07774), LR=5.70e-05
⭐ Epoch   2: Loss=0.0550, pAUC=0.15456 (best=0.15456), LR=8.40e-05
⭐ Epoch   3: Loss=0.0547, pAUC=0.15873 (best=0.15873), LR=1.11e-04
   Epoch   5: Loss=0.0543, pAUC=0.14880 (best=0.15873), LR=1.65e-04
⭐ Epoch   9: Loss=0.0537, pAUC=0.15951 (best=0.15951), LR=2.73e-04
   Epoch  10: Loss=0.0535, pAUC=0.15513 (best=0.15951), LR=3.00e-04
   Epoch  15: Loss=0.0532, pAUC=0.11315 (best=0.15951), LR=2.99e-04
   Epoch  20: Loss=0.0529, pAUC=0.14688 (best=0.15951), LR=2.96e-04
   Epoch  25: Loss=0.0522, pAUC=0.07215 (best=0.15951), LR=2.92e-04
⏹️  Early stopping at epoch 29

✅ Fold 1 Best pAUC: 0.15951

FOLD 2/5
Train size: 313765, Val size: 87294
⭐ Epoch   1: Loss=0.1140, pAUC=0.06838 (best=0.06838), LR=5.70e-05
⭐ Epoch   2: Loss=0.0554, pAUC=0.14254 (best=0.14254), LR=8.40e-05
⭐ Epoch   3: Loss=0.0552, pAUC=0.15518 (best=0.15518), LR=1.11e-04
⭐ Epoch   4: Loss=0.0549, pAUC=0.15531 

## 9. Test Prediction

## 9.1 Test-Time Augmentation (TTA)

TTA cho tabular data bằng cách thêm noise nhỏ vào numerical features

In [14]:
@torch.no_grad()
def predict_with_tta(model, data_loader, device, n_tta=5, tta_noise_std=0.01):
    """
    Test-Time Augmentation for tabular data
    
    Args:
        model: Trained TabM model
        data_loader: DataLoader for test data
        device: cuda/cpu
        n_tta: Number of TTA iterations
        tta_noise_std: Standard deviation of Gaussian noise to add
    
    Returns:
        averaged predictions across TTA iterations
    """
    model.eval()
    all_tta_preds = []
    
    for tta_iter in range(n_tta):
        fold_preds = []
        
        for X_num, X_cat in data_loader:
            X_num, X_cat = X_num.to(device), X_cat.to(device)
            
            # Add Gaussian noise for TTA (except first iteration - original data)
            if tta_iter > 0:
                noise = torch.randn_like(X_num) * tta_noise_std
                X_num_aug = X_num + noise
            else:
                X_num_aug = X_num
            
            # Forward pass
            y_pred = model(X_num_aug, X_cat).squeeze(-1)  # (batch, K)
            
            # Average K predictions
            probs = torch.sigmoid(y_pred).mean(dim=1)
            fold_preds.extend(probs.cpu().numpy())
        
        all_tta_preds.append(fold_preds)
    
    # Average across TTA iterations
    return np.mean(all_tta_preds, axis=0)

print("✅ Test-Time Augmentation function ready")

✅ Test-Time Augmentation function ready


In [15]:
# Create test dataset
test_dataset = ISICDataset(X_num_test, X_cat_test)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], 
                        shuffle=False, num_workers=0, pin_memory=True)

# ============================================
# OPTIMIZED PREDICTION with TTA
# ============================================
print("="*60)
print("GENERATING TEST PREDICTIONS WITH TTA")
print("="*60)

test_predictions = []

for fold, model in enumerate(fold_models):
    print(f"Fold {fold+1}/{N_SPLITS} predictions...", end=" ")
    
    # Use TTA for better predictions
    fold_preds = predict_with_tta(
        model, test_loader, device, 
        n_tta=5,  # 5 TTA iterations
        tta_noise_std=0.01
    )
    
    test_predictions.append(fold_preds)
    print(f"✅ Done (mean pred: {fold_preds.mean():.4f})")

# Average predictions across folds
final_predictions = np.mean(test_predictions, axis=0)

print(f"\n{'='*60}")
print(f"Final predictions shape: {final_predictions.shape}")
print(f"Prediction range: [{final_predictions.min():.4f}, {final_predictions.max():.4f}]")
print(f"Mean prediction: {final_predictions.mean():.4f}")
print(f"Std prediction: {final_predictions.std():.4f}")
print(f"{'='*60}")

GENERATING TEST PREDICTIONS WITH TTA
Fold 1/5 predictions... ✅ Done (mean pred: 0.0307)
Fold 2/5 predictions... ✅ Done (mean pred: 0.0190)
Fold 3/5 predictions... ✅ Done (mean pred: 0.0174)
Fold 4/5 predictions... ✅ Done (mean pred: 0.0323)
Fold 5/5 predictions... ✅ Done (mean pred: 0.0124)

Final predictions shape: (3,)
Prediction range: [0.0115, 0.0404]
Mean prediction: 0.0224
Std prediction: 0.0128


## 10. Create Submission

In [16]:
df_subm[target_col] = final_predictions

df_subm.to_csv('submission.csv')
df_subm.head()

,target
isic_id,
ISIC_0015657,0.015240
ISIC_0015729,0.011473
ISIC_0015740,0.040359
